# Phase 2 — Step 2.2: LSTM Model Training

Trains a bidirectional LSTM on the (N, 30, 1662) landmark sequences.  Outputs a `.h5` checkpoint, training-history plots, and an entry in the experiment log.

**What this notebook does, in plain language:**
1. Load `X.npy` / `y.npy` / `label_map.json` from your Google Drive
2. Compute class weights to compensate for any imbalance
3. Build a bidirectional LSTM with dropout regularisation
4. Train it for up to 100 epochs with early stopping
5. Save the best checkpoint to Drive
6. Plot loss and accuracy curves
7. Append a row to `experiment_log.csv` so you can compare future runs

## Cell 1 — Imports

In [ ]:
import os
import json
import csv
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, utils
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version    : {keras.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available   : {len(gpus)}")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"Using GPU        : {gpus[0].name}")
else:
    print("WARNING: no GPU detected.  Training will be slow.")

**What is an LSTM?**

An LSTM (Long Short-Term Memory) is a type of neural network designed for sequences.  Unlike a regular network that treats each frame independently, an LSTM maintains a **memory** of what it has seen in earlier frames and uses that to influence its prediction on the current frame.

For sign language this matters because the meaning of a sign is in the **motion** across time, not the pose at any single instant.  A still image of a hand at one position looks similar across many signs.  The full 30-frame motion is what makes them different.

We use a **bidirectional** LSTM which reads the sequence both forwards and backwards, so it can also learn from the end of the sign back to the beginning.  This roughly doubles the model parameters but significantly improves accuracy on motion data.

## Cell 2 — Mount Drive and load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/nsl-data'

X = np.load(os.path.join(DATA_DIR, 'X.npy'))
y = np.load(os.path.join(DATA_DIR, 'y.npy'))

with open(os.path.join(DATA_DIR, 'label_map.json')) as f:
    label_map = json.load(f)

idx_to_label = {v: k for k, v in label_map.items()}
n_classes = len(label_map)

print(f"X shape  : {X.shape}   dtype: {X.dtype}")
print(f"y shape  : {y.shape}   dtype: {y.dtype}")
print(f"Classes  : {n_classes}")
print(f"Labels   : {[idx_to_label[i] for i in range(n_classes)]}")

## Cell 3 — Train/val/test split and class weights

In [ ]:
# Stratified split: 70% train, 15% val, 15% test
from sklearn.model_selection import train_test_split

# First split: train vs (val + test)
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42,
)
# Second split: val vs test (each = 15% of total)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42,
)

print(f"Train : {X_train.shape}  labels {X_train.shape[0]}")
print(f"Val   : {X_val.shape}  labels {X_val.shape[0]}")
print(f"Test  : {X_test.shape}  labels {X_test.shape[0]}")

# Class weights to compensate for imbalance.
# compute_class_weight returns one weight per class, higher for rarer classes.
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train,
)
class_weight_dict = {int(c): float(w) for c, w in zip(np.unique(y_train), weights)}
print("\nClass weights (label: weight):")
for label_idx, w in sorted(class_weight_dict.items()):
    print(f"  {idx_to_label[label_idx]:20s}  {w:.2f}")

# Save the splits to Drive for the evaluation notebook
SPLIT_DIR = '/content/drive/MyDrive/nsl-splits'
os.makedirs(SPLIT_DIR, exist_ok=True)
np.save(os.path.join(SPLIT_DIR, 'X_train.npy'), X_train)
np.save(os.path.join(SPLIT_DIR, 'X_val.npy'),   X_val)
np.save(os.path.join(SPLIT_DIR, 'X_test.npy'),  X_test)
np.save(os.path.join(SPLIT_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(SPLIT_DIR, 'y_val.npy'),   y_val)
np.save(os.path.join(SPLIT_DIR, 'y_test.npy'),  y_test)
print(f"\nSaved splits to {SPLIT_DIR}")

**Why class weights?**

If class A has 30 samples and class B has 5 samples, a naive training will correctly predict A 6 times more often than B just by always guessing A.  The loss function would still go down even though the model has learned nothing useful about class B.

Class weights multiply the loss for each class by a factor inversely proportional to its frequency.  The model is then penalised more for getting a rare class wrong, forcing it to actually learn to recognise it.

## Cell 4 — Build the model

In [ ]:
def build_model(input_shape, n_classes):
    inputs = keras.Input(shape=input_shape, name='landmarks')

    # Mask the all-zero frames that MediaPipe produces when it fails
    # to detect a body part.  This tells the LSTM to ignore those
    # timesteps instead of treating them as valid data.
    non_zero = layers.Lambda(
        lambda x: tf.cast(
            tf.reduce_any(tf.not_equal(x, 0.0), axis=-1, keepdims=True),
            tf.float32,
        ),
        name='non_zero_mask',
    )(inputs)

    # Normalise landmark coordinates to roughly zero mean and unit
    # variance.  Helps the optimiser converge.
    norm = layers.BatchNormalization(name='batch_norm')(inputs)

    # First bidirectional LSTM.  return_sequences=True because the
    # second LSTM needs a sequence, not just a final state.
    x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=True, dropout=0.3),
        name='lstm_1',
    )(norm)

    # Second bidirectional LSTM.  This time we only keep the final
    # state — a single 128-d vector summarising the whole sequence.
    x = layers.Bidirectional(
        layers.LSTM(64, dropout=0.3),
        name='lstm_2',
    )(x)

    # Dense head with dropout.  The Dropout layer randomly zeroes
    # 50% of the activations during training, which prevents the
    # network from relying on any single feature too much.
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(n_classes, activation='softmax', name='predictions')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='nsl_lstm')
    return model

model = build_model(input_shape=(30, 1662), n_classes=n_classes)
model.summary()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

**The architecture, layer by layer:**

- **Input layer** — accepts a 30-frame sequence of 1662 landmarks per frame
- **Batch normalization** — rescales the inputs so they have roughly zero mean and unit variance, which helps the optimiser converge faster
- **LSTM 1 (bidirectional, 64 units, return_sequences=True)** — reads the sequence and outputs a 128-d vector at every time step (because bidirectional, 64 + 64 = 128)
- **LSTM 2 (bidirectional, 64 units)** — reads the output of LSTM 1 and produces a single 128-d vector summarising the entire sequence
- **Dropout 0.5** — randomly zeroes 50% of activations during training to prevent the model from memorising
- **Dense 32 with ReLU** — a small fully-connected layer to combine the LSTM output into something the final classifier can use
- **Dropout 0.5** — second dropout
- **Dense n_classes with softmax** — produces a probability for each sign; the highest one is the prediction

## Cell 5 — Set up callbacks

In [ ]:
CHECKPOINT_DIR = '/content/drive/MyDrive/nsl-checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_model.h5')

cbs = [
    # Save the best model (lowest val_loss) to Drive
    callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_loss',
        save_best_only=True,
        verbose=1,
    ),
    # Stop if val_loss does not improve for 15 epochs
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1,
    ),
    # Reduce learning rate when val_loss plateaus
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1,
    ),
]
print(f"Checkpoint will be saved to: {checkpoint_path}")

**What each callback does:**

- **ModelCheckpoint** — every time `val_loss` improves, it saves the model weights to Drive.  If the session disconnects partway through training, you do not lose progress.
- **EarlyStopping** — if `val_loss` does not improve for 15 consecutive epochs, training halts and the best weights are restored.  Prevents wasting time on a model that has stopped learning.
- **ReduceLROnPlateau** — when training stalls, automatically halves the learning rate.  Often the difference between converging to a good solution and getting stuck in a local minimum.

## Cell 6 — Train

In [ ]:
EPOCHS = 100
BATCH_SIZE = 16

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=cbs,
    verbose=1,
)
print("\nTraining complete.")

## Cell 7 — Plot training history

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['loss'],     label='train loss')
ax1.plot(history.history['val_loss'], label='val loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss over epochs')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['accuracy'],     label='train acc')
ax2.plot(history.history['val_accuracy'], label='val acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy over epochs')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/nsl-checkpoints/training_curves.png',
            dpi=100, bbox_inches='tight')
plt.show()

**How to read these plots:**

- **Train loss going down, val loss flat or going up** = overfitting.  The model is memorising the training data instead of learning the pattern.
- **Both losses going down together** = healthy training.  The model is learning something that generalises.
- **Both losses flat from the start** = model is not learning.  Often a bug (wrong loss function, learning rate too high, labels shuffled).
- **Train accuracy much higher than val accuracy** = overfitting.  The val accuracy is the number to watch — that is the model performance on data it has never seen.

## Cell 8 — Quick test-set evaluation

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_acc:.4f}")

# per-class accuracy on the test set
y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
print("\nPer-class accuracy on test set:")
for label_idx in range(n_classes):
    mask = (y_test == label_idx)
    if mask.sum() == 0:
        continue
    acc = (y_pred[mask] == label_idx).mean()
    print(f"  {idx_to_label[label_idx]:20s}  "
          f"{mask.sum()} samples, {acc:.0%} correct")

## Cell 9 — Append to experiment log

In [ ]:
LOG_PATH = '/content/drive/MyDrive/nsl-checkpoints/experiment_log.csv'
os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)

best_epoch = int(np.argmin(history.history['val_loss']) + 1)
best_val_loss = float(np.min(history.history['val_loss']))
best_val_acc  = float(np.max(history.history['val_accuracy']))
n_params = model.count_params()

row = {
    "timestamp":     pd.Timestamp.utcnow().isoformat(),
    "architecture":  "BiLSTM(64)-BiLSTM(64)-Dense(32)",
    "hidden_units":  64,
    "dropout":       0.5,
    "learning_rate": 1e-3,
    "batch_size":    BATCH_SIZE,
    "epochs_run":    len(history.history['loss']),
    "best_epoch":    best_epoch,
    "val_loss":      round(best_val_loss, 4),
    "val_accuracy":  round(best_val_acc,  4),
    "test_loss":     round(float(test_loss), 4),
    "test_accuracy": round(float(test_acc),  4),
    "n_params":      n_params,
    "n_samples":     int(X.shape[0]),
    "n_classes":     n_classes,
    "notes":         "first run on small dataset (92 samples)",
}

file_exists = os.path.exists(LOG_PATH)
with open(LOG_PATH, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(row.keys()))
    if not file_exists:
        writer.writeheader()
    writer.writerow(row)
print(f"Appended to {LOG_PATH}")
print(pd.read_csv(LOG_PATH).to_string())

## Cell 10 — Download the model to your laptop

In [ ]:
from google.colab import files

# Refresh: reload the best checkpoint (EarlyStopping restored it)
model.save('/content/best_model.h5')

print(f"Final test accuracy: {test_acc:.4f}")
print(f"\nDownloading best_model.h5 to your laptop...")
files.download('/content/best_model.h5')

print("\nNext steps:")
print("  1. Save best_model.h5 to model/checkpoints/ in your repo")
print("  2. Open 03_evaluation.ipynb for the full confusion matrix and report")
print("  3. Open 04_tflite_conversion.ipynb to export the mobile model")